In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



    P[B,C,D,F]start, the time at which the pulse rises to 20% of its peak with respect to the Channel A;

    P[A,B,C,D,F]rise, the time it takes for a pulse to rise from 20% to 50% of its peak;

    P[A,B,C,D,F]width, the width (in seconds) of the pulse at 80% of the pulse height;

    P[A,B,C,D,F]fall, the time it takes for a pulse to fall from 40% to 20% of its peak.


In [33]:
full = pd.read_csv("../full_dataset/full_data_revised.csv")
full = full.sort_values(by=['Row'],ascending=True).reset_index(drop=True)
full_copy = full.copy()

In [34]:
out = pd.read_csv("../reduced_dataset/reduced_data.csv").sort_values(by=['Row'],ascending=True).reset_index(drop=True)
onlynum = np.array(full_copy.drop(columns=['PAamp','PBamp','PCamp','PDamp','PFamp','Row','E.76']))

In [54]:
import numpy as np
import random
import multiprocessing as mp
import time
import sys
import math

def get_v_score(idx_list, X):
    if not idx_list: return 0
    cols = X[:, idx_list]
    sorted_cols = cols[:, np.argsort(np.mean(cols, axis=0))]
    return np.sum(np.diff(sorted_cols, axis=1) <= 0)

def annealing_worker(data, n_groups, cols_per_group, seed, result_queue, progress_dict):
    X = np.array(data)
    random.seed(seed)
    np.random.seed(seed)
    
    all_indices = list(range(X.shape[1]))
    random.shuffle(all_indices)
    groups = [all_indices[i*cols_per_group : (i+1)*cols_per_group] for i in range(n_groups)]
    
    current_v = sum(get_v_score(g, X) for g in groups)
    best_v = current_v
    iteration = 0
    stagnation = 0
    temp = 1.0 

    while True:
        iteration += 1
        stagnation += 1
        
        temp = max(0.01, 2.0 * (0.9999 ** iteration)) 
        
        g1, g2 = random.sample(range(n_groups), 2)
        c1, c2 = random.randint(0, cols_per_group-1), random.randint(0, cols_per_group-1)
        
        v_pre = get_v_score(groups[g1], X) + get_v_score(groups[g2], X)
        groups[g1][c1], groups[g2][c2] = groups[g2][c2], groups[g1][c1]
        v_post = get_v_score(groups[g1], X) + get_v_score(groups[g2], X)
        
        delta = v_pre - v_post
        if delta > 0 or (temp > 0.01 and random.random() < math.exp(delta / temp)):
            current_v = current_v - v_pre + v_post
            if current_v < best_v:
                best_v = current_v
                progress_dict[seed] = best_v
                stagnation = 0
                
                try:
                    np.save('best_indices_checkpoint.npy', np.array(groups, dtype=object))
                except:
                    pass
        else:
            # Revert the swap
            groups[g1][c1], groups[g2][c2] = groups[g2][c2], groups[g1][c1]

        if stagnation > 5000:
            for _ in range(5):
                ga, gb = random.sample(range(n_groups), 2)
                ra, rb = random.randint(0, 15), random.randint(0, 15)
                groups[ga][ra], groups[gb][rb] = groups[gb][rb], groups[ga][ra]
            current_v = sum(get_v_score(g, X) for g in groups)
            stagnation = 0

        if current_v == 0:
            result_queue.put(groups)
            break

def parallel_annealing_with_progress(data, n_groups=5, cols_per_group=16):
    X = np.array(data)
    num_cores = mp.cpu_count()
    
    manager = mp.Manager()
    progress_dict = manager.dict()
    result_queue = mp.Queue()
    
    processes = []
    for i in range(num_cores):
        progress_dict[i] = 999999
        p = mp.Process(target=annealing_worker, args=(data, n_groups, cols_per_group, i, result_queue, progress_dict))
        p.start()
        processes.append(p)
    
    print(f"Annealing started on {num_cores} cores. Hunting for 0 violations...")
    
    start_time = time.time()
    try:
        while result_queue.empty():
            elapsed = time.strftime("%H:%M:%S", time.gmtime(time.time() - start_time))
            scores = [progress_dict[i] for i in range(num_cores)]
            global_best = min(scores)
            
            status = f"[{elapsed}] Best: {global_best} | Cores: {scores}"
            sys.stdout.write('\r' + status)
            sys.stdout.flush()
            
            time.sleep(2)
            
    except KeyboardInterrupt:
        for p in processes: p.terminate()
        return None

    winning_groups = result_queue.get()
    for p in processes: p.terminate()
    
    print(f"\n\n--- SUCCESS! Solution found in {time.time() - start_time:.2f} seconds ---")
    
    final_reordered_matrix = np.zeros_like(X)
    for i in range(n_groups):
        g_indices = winning_groups[i]
        cols = X[:, g_indices]
        sort_idx = np.argsort(np.mean(cols, axis=0))
        sorted_cluster = cols[:, sort_idx]
        final_reordered_matrix[:, i*16:(i+1)*16] = sorted_cluster

    return final_reordered_matrix

if __name__ == '__main__':
    final_data = parallel_annealing_with_progress(onlynum)

Annealing started on 32 cores. Hunting for 0 violations...
[00:56:13] Best: 10 | Cores: [np.int64(5909), np.int64(10), np.int64(925), np.int64(7024), np.int64(7616), np.int64(7788), np.int64(615), np.int64(5960), np.int64(925), np.int64(307), np.int64(4723), np.int64(5176), np.int64(4845), np.int64(100), np.int64(10), np.int64(5088), np.int64(3150), np.int64(4603), np.int64(6258), np.int64(166), np.int64(2824), np.int64(957), np.int64(1799), np.int64(10331), np.int64(132), np.int64(615), np.int64(963), np.int64(6390), np.int64(4838), np.int64(6382), np.int64(5121), np.int64(396)]]])]]]]]]]]]]]]]]]]]]]]])]]]]]]]]]]]])]]]]]]]]])]

--- SUCCESS! Solution found in 3375.15 seconds ---


In [105]:
appends = full_copy[['Row','E.76','PAamp','PBamp','PCamp','PDamp','PFamp']].copy()
final2 = pd.DataFrame(final_data.copy())

In [106]:
assembled = pd.concat([appends,final2],axis=1)

In [110]:
assembled.to_csv("data_rearranged.csv",index=False)